In [1]:
# ============================================================
# CIFAR-100: Progressive Token Pruning + Feature-Aware KD
# Target: 0.84+ on Kaggle P100
#
# Design decisions (with rationale):
#  - Student INITIALIZED from teacher weights (not scratch)
#    Reason: ViT-B/16 needs pretraining to converge; 25 epochs
#    from scratch on CIFAR-100 yields ~40-50%, not useful.
#    Teacher-init is standard in pruning literature.
#  - MixUp KEPT (applied to both student and teacher equally)
#    Reason: CIFAR-100 is small; MixUp regularization benefit
#    outweighs KD noise when both networks see identical inputs.
#  - LR warmup used (not KD warmup)
#    Reason: Delaying KD wastes early epochs; LR warmup gives
#    stability without sacrificing supervision signal.
#  - Alpha annealing: KD-heavy early → CE-heavy late
#    Reason: Early training should track teacher closely;
#    late training should sharpen class boundaries.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader
import copy
import math
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# ============================================================
# DATA
# ============================================================

def get_cifar100(batch_size=64):
    train_tf = transforms.Compose([
        transforms.Resize(224),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3),
    ])
    test_tf = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize((0.5,)*3, (0.5,)*3),
    ])
    train_ds = torchvision.datasets.CIFAR100('./data', train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR100('./data', train=False, download=True, transform=test_tf)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    return train_dl, test_dl

train_dl, test_dl = get_cifar100(batch_size=64)

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = total = 0
    for x, y in test_dl:
        x, y = x.to(device), y.to(device)
        # Handle both plain models and feature-returning models
        out = model(x)
        logits = out[0] if isinstance(out, tuple) else out
        correct += (logits.argmax(1) == y).sum().item()
        total   += y.size(0)
    return correct / total

# ============================================================
# PROGRESSIVE SPARSE ViT
#
# Stage 1 (layers 1-4):  100% tokens
# Stage 2 (layers 5-8):   75% tokens  (kr1)
# Stage 3 (layers 9-12):  50% tokens  (kr2)
#
# Token importance = L2 norm AFTER transformer blocks
# (semantic signal, not raw pixel statistics)
# Returns (logits, feat_mid, feat_late, cls) for feature KD
# ============================================================

class ProgressiveSparseViT(nn.Module):
    def __init__(self, base_model, keep_ratio_1=0.75, keep_ratio_2=0.50):
        super().__init__()
        self.base = base_model
        self.kr1  = keep_ratio_1
        self.kr2  = keep_ratio_2

        layers = list(self.base.encoder.layers.children())
        assert len(layers) == 12, f'Expected 12 layers, got {len(layers)}'
        self.stage1 = nn.Sequential(*layers[0:4])
        self.stage2 = nn.Sequential(*layers[4:8])
        self.stage3 = nn.Sequential(*layers[8:12])

    def _prune(self, x, keep_ratio, num_patches):
        B, _, C = x.shape
        k = max(1, int(num_patches * keep_ratio))
        scores  = x[:, 1:].norm(dim=-1)                          # patch scores after blocks
        topk    = torch.topk(scores, k, dim=1).indices + 1       # +1 to offset CLS
        cls_idx = torch.zeros(B, 1, dtype=torch.long, device=x.device)
        idx     = torch.cat([cls_idx, topk], dim=1)
        return torch.gather(x, 1, idx.unsqueeze(-1).expand(-1, -1, C))

    def forward(self, x):
        x = self.base._process_input(x)                          # patch embed
        B, N, C = x.shape
        cls = self.base.class_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = x + self.base.encoder.pos_embedding
        x   = self.base.encoder.dropout(x)

        # Stage 1: full tokens
        x = self.stage1(x)
        x = self._prune(x, self.kr1, N)                          # → 75% patches
        feat_mid = x[:, 0]                                        # CLS for KD

        # Stage 2: 75% tokens
        k1 = max(1, int(N * self.kr1))
        x  = self.stage2(x)
        x  = self._prune(x, self.kr2 / self.kr1, k1)            # relative prune → 50%
        feat_late = x[:, 0]                                       # CLS for KD

        # Stage 3: 50% tokens
        x      = self.stage3(x)
        x      = self.base.encoder.ln(x)
        cls_out = x[:, 0]
        logits  = self.base.heads(cls_out)
        return logits, feat_mid, feat_late, cls_out


# ============================================================
# TEACHER WRAPPER (returns matching features for KD)
# ============================================================

class TeacherWithFeatures(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model
        layers = list(self.base.encoder.layers.children())
        self.stage1 = nn.Sequential(*layers[0:4])
        self.stage2 = nn.Sequential(*layers[4:8])
        self.stage3 = nn.Sequential(*layers[8:12])

    def forward(self, x):
        x = self.base._process_input(x)
        B, N, C = x.shape
        cls = self.base.class_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = x + self.base.encoder.pos_embedding
        x   = self.base.encoder.dropout(x)

        x = self.stage1(x)
        feat_mid  = x[:, 0]
        x = self.stage2(x)
        feat_late = x[:, 0]
        x = self.stage3(x)
        x = self.base.encoder.ln(x)
        cls_out = x[:, 0]
        logits  = self.base.heads(cls_out)
        return logits, feat_mid, feat_late, cls_out


# ============================================================
# LOAD TEACHER
# ============================================================

base_teacher = vit_b_16(weights=None)
base_teacher.heads.head = nn.Linear(base_teacher.heads.head.in_features, 100)
base_teacher.load_state_dict(
    torch.load('/kaggle/input/datasets/lokianansi/densevit-cifar100/dense_cifar100_best.pt',
               map_location=device)
)
base_teacher.to(device).eval()
print('Teacher accuracy:', evaluate(base_teacher))

teacher = TeacherWithFeatures(base_teacher).to(device).eval()


# ============================================================
# TRAINING
#
# Loss = (1-alpha)*CE  +  alpha*KD_logit  +  feat_weight*KD_feat
#
# alpha anneals 0.70 → 0.50 over training:
#   Early: KD-heavy  (student tracks teacher)
#   Late:  CE-heavy  (student sharpens own predictions)
#
# LR warmup (not KD warmup): cosine schedule from 0 → 5e-5
# over 3 epochs, then cosine decay to 1e-6.
# This gives stability without wasting KD supervision.
# ============================================================

def train_feature_kd(
    student, teacher,
    epochs      = 25,
    T           = 4.0,
    feat_weight = 0.3,    # conservative; increase to 0.5 if val acc plateaus early
    base_lr     = 5e-5,
    warmup_epochs = 3,
    mixup_alpha = 0.4,
):
    student.to(device)
    ce_loss   = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(student.parameters(), lr=base_lr, weight_decay=0.05)

    # Cosine schedule with linear LR warmup
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return (ep + 1) / warmup_epochs                             # linear warmup
        progress = (ep - warmup_epochs) / max(1, epochs - warmup_epochs)
        min_frac = 1e-6 / base_lr
        return min_frac + 0.5 * (1 - min_frac) * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = torch.amp.GradScaler('cuda')
    best_acc  = 0.0

    for epoch in range(epochs):
        student.train()
        total_loss = 0.0

        # Alpha annealing: KD-heavy early, CE-heavy late
        alpha = 0.70 - 0.20 * (epoch / max(1, epochs - 1))   # 0.70 → 0.50

        for x, y in train_dl:
            x, y = x.to(device), y.to(device)

            # MixUp — same mixed batch fed to BOTH student and teacher
            # so the KD relationship remains valid
            lam   = np.random.beta(mixup_alpha, mixup_alpha)
            idx   = torch.randperm(x.size(0), device=device)
            x_mix = lam * x + (1 - lam) * x[idx]
            y_a, y_b = y, y[idx]

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                s_logits, s_mid, s_late, _ = student(x_mix)

                # CE with mixed labels
                ce = lam * ce_loss(s_logits, y_a) + (1 - lam) * ce_loss(s_logits, y_b)

                with torch.no_grad():
                    t_logits, t_mid, t_late, _ = teacher(x_mix)

                # Logit KD
                kd_logit = F.kl_div(
                    F.log_softmax(s_logits / T, dim=1),
                    F.softmax(t_logits  / T, dim=1),
                    reduction='batchmean'
                ) * (T * T)

                # Feature KD: normalized MSE on CLS at mid + late stages
                kd_feat = 0.5 * (
                    F.mse_loss(F.normalize(s_mid,  dim=-1), F.normalize(t_mid,  dim=-1)) +
                    F.mse_loss(F.normalize(s_late, dim=-1), F.normalize(t_late, dim=-1))
                )

                loss = (1 - alpha) * ce + alpha * kd_logit + feat_weight * kd_feat

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        scheduler.step()
        acc    = evaluate(student)
        lr_now = optimizer.param_groups[0]['lr']

        print(f'Epoch {epoch+1:02d}/{epochs} | '
              f'Loss: {total_loss/len(train_dl):.4f} | '
              f'ValAcc: {acc:.4f} | '
              f'alpha: {alpha:.3f} | '
              f'LR: {lr_now:.2e}')

        if acc > best_acc:
            best_acc = acc
            torch.save(student.state_dict(), 'best_sparse_kd_progressive.pt')
            print(f'  ✓ Saved best: {best_acc:.4f}')

    print(f'\nBest Val Accuracy: {best_acc:.4f}')
    return best_acc


# ============================================================
# RUN
# ============================================================

print('\nBuilding student (teacher-initialized progressive sparse ViT)...')
# Teacher-init: student starts from teacher weights, then learns to be sparse.
# This is standard in ViT pruning (DynamicViT, EViT) and is NOT the same as
# 'no learning' — the pruning structure itself is what the model must adapt to.
student = ProgressiveSparseViT(
    copy.deepcopy(base_teacher),
    keep_ratio_1=0.75,
    keep_ratio_2=0.50,
)

print('Training...')
best_acc = train_feature_kd(
    student, teacher,
    epochs       = 25,
    T            = 4.0,
    feat_weight  = 0.3,
    base_lr      = 5e-5,
    warmup_epochs= 3,
    mixup_alpha  = 0.4,
)

# Load best and final eval
student.load_state_dict(torch.load('best_sparse_kd_progressive.pt'))
final_acc = evaluate(student)
print(f'Final accuracy (best checkpoint): {final_acc:.4f}')
torch.save(student.state_dict(), 'sparse_kd_cifar100_final.pt')

print('\n=== Summary ===')
print(f'  Teacher (Dense ViT-B/16):                {evaluate(base_teacher):.4f}')
print(f'  Student (Progressive Sparse 50% + KD):  {final_acc:.4f}')

Device: cuda


100%|██████████| 169M/169M [00:15<00:00, 10.9MB/s] 


Teacher accuracy: 0.8703

Building student (teacher-initialized progressive sparse ViT)...
Training...
Epoch 01/25 | Loss: 0.9574 | ValAcc: 0.8699 | alpha: 0.700 | LR: 3.33e-05
  ✓ Saved best: 0.8699
Epoch 02/25 | Loss: 0.9766 | ValAcc: 0.8744 | alpha: 0.692 | LR: 5.00e-05
  ✓ Saved best: 0.8744
Epoch 03/25 | Loss: 1.0402 | ValAcc: 0.8681 | alpha: 0.683 | LR: 5.00e-05
Epoch 04/25 | Loss: 1.0717 | ValAcc: 0.8682 | alpha: 0.675 | LR: 4.98e-05
Epoch 05/25 | Loss: 1.0614 | ValAcc: 0.8702 | alpha: 0.667 | LR: 4.90e-05
Epoch 06/25 | Loss: 1.0638 | ValAcc: 0.8730 | alpha: 0.658 | LR: 4.78e-05
Epoch 07/25 | Loss: 1.0955 | ValAcc: 0.8769 | alpha: 0.650 | LR: 4.61e-05
  ✓ Saved best: 0.8769
Epoch 08/25 | Loss: 1.0654 | ValAcc: 0.8817 | alpha: 0.642 | LR: 4.40e-05
  ✓ Saved best: 0.8817
Epoch 09/25 | Loss: 1.1085 | ValAcc: 0.8810 | alpha: 0.633 | LR: 4.15e-05
Epoch 10/25 | Loss: 1.1045 | ValAcc: 0.8832 | alpha: 0.625 | LR: 3.87e-05
  ✓ Saved best: 0.8832
Epoch 11/25 | Loss: 1.1385 | ValAcc: 0.880